# TAEL V2 (`v2.0.0-rc1`) — Quickstart & Dynamic Task Router

This notebook demonstrates:
1. **TAEL's Dynamic Task Router (`tael_router.py`)** classifying prompts across 9 software engineering domains.
2. **Confidence scoring, domain-specific sampling presets (`temperature`, `top_p`), and targeted tool filtering.**
3. **Live inference via the local Ollama runtime (`tael-agent`).**

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from tael_router import route_task

sample_prompts = [
    "Write a Python function `two_sum(nums, target)` in O(n) time using a dictionary.",
    "SELECT c.name, SUM(o.amount) FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.name;",
    "Refactor this deeply nested function using Guard Clauses and early returns.",
    "Fix this Traceback (most recent call last): TypeError: 'NoneType' object is not subscriptable",
    "How do I squash the last 3 commits using git rebase -i HEAD~3?",
    "Design a high-availability microservices architecture using Apache Kafka and Redis deduplication.",
    "In the src folder, create a file config.py and run pytest in the terminal.",
    "Calculate the derivative of f(x) = 3x^4 - 5x^2 + 2x - 7 and solve for f'(1).",
    "Hello TAEL! What can you help me with today?"
]

print(f"{'Category':<13} | {'Confidence':<10} | {'Temp':<4} | {'MaxTok':<6} | Tools")
print("-" * 75)
for p in sample_prompts:
    decision = route_task(p)
    print(f"{decision.category.value:<13} | {decision.confidence*100:>9.1f}% | {decision.temperature:<4.2f} | {decision.max_tokens:<6} | {decision.tool_names[:5]}")

Category      | Confidence | Temp | MaxTok | Tools
---------------------------------------------------------------------------
Programming   |      81.0% | 0.15 |   2048 | ['read_file', 'write_file', 'search_text', 'execute_python', 'finish']
SQL           |      99.0% | 0.10 |   1024 | ['read_file', 'search_text', 'execute_python', 'finish']
Refactoring   |      84.0% | 0.20 |   2048 | ['read_file', 'write_file', 'search_text', 'finish']
Debugging     |      93.0% | 0.20 |   2048 | ['read_file', 'write_file', 'search_text', 'execute_python', 'git_diff', 'finish']
Git           |      93.0% | 0.10 |   1024 | ['git_status', 'git_diff', 'git_log', 'git_branch', 'read_file', 'finish']
Architecture  |      99.0% | 0.35 |   2048 | ['read_file', 'list_dir', 'search_text', 'finish']
Tool Use      |      89.0% | 0.15 |   2048 | ['read_file', 'write_file', 'list_dir', 'search_text', 'git_status', ...]
Math          |      99.0% | 0.00 |   1536 | ['execute_python', 'finish']
General Chat  |     

## Querying Local Ollama Runtime (`tael-agent`) with Routed Parameters

In [2]:
import json
import urllib.request

prompt = "Write an SQLite query to find customer name and total amount spent for customers spending > 100 from tables customers(id, name) and orders(id, customer_id, amount)."
decision = route_task(prompt)
print(f"Routed Category: {decision.category.value} (Confidence: {decision.confidence:.0%}, Temperature: {decision.temperature})")

payload = json.dumps({
    "model": "tael-agent",
    "messages": [
        {"role": "system", "content": decision.system_prompt},
        {"role": "user", "content": prompt}
    ],
    "stream": False,
    "options": {
        "temperature": decision.temperature,
        "top_p": decision.top_p,
        "num_ctx": 4096
    }
}).encode("utf-8")

req = urllib.request.Request("http://127.0.0.1:11434/api/chat", data=payload, headers={"Content-Type": "application/json"})
with urllib.request.urlopen(req, timeout=30) as resp:
    data = json.loads(resp.read().decode("utf-8"))
    print("Response:\n" + data["message"]["content"])

Routed Category: SQL (Confidence: 99%, Temperature: 0.1)
Response:
SELECT c.name, SUM(o.amount) AS total_spent
FROM customers c
JOIN orders o ON c.id = o.customer_id
GROUP BY c.name
HAVING total_spent > 100
ORDER BY total_spent DESC;
